# Customer history features

What the customer did before this order. Every value uses only strictly earlier orders (`past_rate` freezes history at the start of each day), so nothing leaks from the order being scored or from same-day siblings.

In [1]:
import sys; sys.path.append("..")
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from src.data import INTERIM, load_orders, save_features
from src.features import past_rate

## Orders

**Load the order table in time order.** `mergesort` keeps the sort stable, which matters because every past-only calculation below depends on deterministic ordering of ties.

In [2]:
orders = pd.read_parquet(INTERIM / "orders.parquet").sort_values("orderDate", kind="mergesort")
orders.shape

(738698, 13)

## Past behaviour

**The leakage-critical cell.** `past_rate` returns each customer's return rate over *strictly earlier* orders, frozen at the start of the day so same-day orders cannot see each other. `shift(1)` inside the customer group gives the previous order's date and value; `.shift(1).expanding().mean()` gives the running average of prior orders only. Verify it worked by checking that `days_since_prev_order` is null exactly once per customer — 311,369 nulls for 311,369 customers.

In [3]:
rate, n = past_rate(orders, "customerID", "orderDate", "any_return")
orders["cust_past_return_rate"] = rate
orders["cust_prior_orders"] = n

g = orders.groupby("customerID", sort=False)
orders["days_since_prev_order"] = (orders.orderDate - g.orderDate.shift(1)).dt.days
orders["cust_prev_order_value"] = g.order_value.shift(1)
orders["cust_mean_prev_value"] = g.order_value.transform(lambda s: s.shift(1).expanding().mean())
orders[["cust_past_return_rate", "cust_prior_orders", "days_since_prev_order"]].describe().round(3)

,cust_past_return_rate,cust_prior_orders,days_since_prev_order
count,738698.000,738698.000,427329.000
mean,0.636,4.362,68.929
std,0.033,11.918,92.042
min,0.169,0.000,0.000
25%,0.635,0.000,8.000
50%,0.635,1.000,31.000
75%,0.642,4.000,94.000
max,0.945,363.000,633.000


## Save

**Persist only the derived columns.** The order-level fields came from `orders.parquet` and would duplicate on merge, so only the five new history features are written, keyed on `orderID`. Note all five are null-safe: a first-time customer legitimately has no previous order, and LightGBM reads that NaN as information rather than an error.

In [4]:
cols = ["orderID", "cust_past_return_rate", "cust_prior_orders", "days_since_prev_order",
        "cust_prev_order_value", "cust_mean_prev_value"]
save_features(orders[cols], "customer", keys=("orderID",))
orders[cols].tail(50)

,orderID,cust_past_return_rate,cust_prior_orders,days_since_prev_order,cust_prev_order_value,cust_mean_prev_value
738643,a1744127,0.610682,2,127.0,45.990002,71.985003
738644,a1744128,0.635109,0,NaN,NaN,NaN
738645,a1744129,0.662377,7,209.0,169.000000,92.274286
738646,a1744130,0.635109,0,NaN,NaN,NaN
738647,a1744131,0.642264,1,5.0,39.990002,39.990002
738648,a1744132,0.642264,1,418.0,80.000000,80.000000
738649,a1744133,0.635109,0,NaN,NaN,NaN
738650,a1744134,0.631918,5,22.0,417.880005,181.554001
738651,a1744135,0.753115,32,17.0,269.950012,194.736251
738652,a1744136,0.635109,0,NaN,NaN,NaN
